# 🔴 Solution: Capsule-Convex Polygon Overlap

**Primitives:** convex point-in-polygon (cross-product signs), orientation test (segment-segment crossing), parametric segment-to-segment distance

**Reduction:** overlap is True iff (1) either spine endpoint is inside the polygon, OR (2) the spine crosses any polygon edge, OR (3) the minimum distance from the spine to any polygon edge is ≤ r. Cases 1 and 2 can be unified: if either fires, the distance to the polygon region is 0. Case 3 covers the near-miss where the spine stays outside but is within reach.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitives: cross-product sign, orientation test, parametric segment distance

def capsule_poly_overlap(capsule, poly):
    p1 = capsule[:2]; p2 = capsule[2:4]; r = capsule[4]
    v0 = poly                               # (V, 2) edge starts
    v1 = np.roll(poly, -1, axis=0)          # (V, 2) edge ends
    edges = v1 - v0                         # (V, 2)

    # ── Case 1: endpoint inside convex polygon ─────────────────────────────
    # CCW polygon: point P is inside iff cross(edge_i, P-v_i) >= 0 for all i
    def inside(pt):
        vecs  = pt - v0                     # (V, 2)
        cross = edges[:,0]*vecs[:,1] - edges[:,1]*vecs[:,0]  # (V,)
        return bool((cross >= 0).all())

    if inside(p1) or inside(p2):
        return True

    # ── Case 2 + 3: check each polygon edge ───────────────────────────────
    # Vectorised over V edges simultaneously
    EPS = 1e-12
    spine = p2 - p1                         # (2,)
    d1 = spine[None, :]                     # (1, 2) — spine direction
    d2 = edges                              # (V, 2) — edge directions
    rv = p1[None, :] - v0                   # (V, 2)

    a = (d1 * d1).sum(-1)                   # (1,)
    e = (d2 * d2).sum(-1)                   # (V,)
    b = (d1 * d2).sum(-1)                   # (V,)
    c = (d1 * rv).sum(-1)                   # (V,)
    f = (d2 * rv).sum(-1)                   # (V,)

    denom = a * e - b * b                   # (V,)

    s = np.where(
        denom > EPS,
        np.clip((b * f - c * e) / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
        0.0
    )
    t = np.clip((b * s + f) / np.where(e > EPS, e, 1.0), 0.0, 1.0)
    s = np.clip((b * t - c) / np.where(a > EPS, a, 1.0), 0.0, 1.0)

    cp1 = p1[None, :] + s[:, None] * d1    # (V, 2)
    cp2 = v0 + t[:, None] * d2             # (V, 2)
    dists = np.sqrt(((cp1 - cp2) ** 2).sum(-1))  # (V,)

    return bool((dists <= r).any())

In [ ]:
# 🔍 Verify solution
square = np.array([[1.0,0.0],[3.0,0.0],[3.0,2.0],[1.0,2.0]])
cap1 = np.array([0.0, 1.0, 4.0, 1.0, 0.5])
cap2 = np.array([0.0, 5.0, 4.0, 5.0, 0.3])
print("crosses:", capsule_poly_overlap(cap1, square))  # expect True
print("far away:", capsule_poly_overlap(cap2, square)) # expect False

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: spine entirely inside polygon ─────────────────────────────────
sq = np.array([[0.0,0.0],[4.0,0.0],[4.0,4.0],[0.0,4.0]])
cap1 = np.array([1.0, 1.0, 3.0, 3.0, 0.2])
assert capsule_poly_overlap(cap1, sq) == True, "Spine inside polygon → overlap"
print("Test 1 passed: spine inside polygon")

# ── Test 2: spine crosses polygon boundary ────────────────────────────────
sq2 = np.array([[0.0,0.0],[2.0,0.0],[2.0,2.0],[0.0,2.0]])
cap2 = np.array([-1.0, 1.0, 3.0, 1.0, 0.1])
assert capsule_poly_overlap(cap2, sq2) == True, "Spine crossing boundary → overlap"
print("Test 2 passed: spine crosses boundary")

# ── Test 3: capsule fully separated, r too small ──────────────────────────
cap3 = np.array([4.0, 0.5, 6.0, 0.5, 0.5])
assert capsule_poly_overlap(cap3, sq2) == False, "Separated capsule → no overlap"
print("Test 3 passed: fully separated")

# ── Test 4: radius just reaches a polygon vertex ──────────────────────────
tri = np.array([[0.0,0.0],[2.0,0.0],[1.0,2.0]])
cap_reach = np.array([0.5, -1.0, 1.5, -1.0, 1.2])
cap_miss  = np.array([0.5, -1.0, 1.5, -1.0, 0.9])
assert capsule_poly_overlap(cap_reach, tri) == True,  "r reaches vertex → overlap"
assert capsule_poly_overlap(cap_miss,  tri) == False, "r too small → no overlap"
print("Test 4 passed: radius near vertex")

# ── Test 5: 50-pair reference comparison + timing ─────────────────────────
def _inside_convex(pt, poly):
    edges = np.roll(poly, -1, axis=0) - poly
    vecs  = pt - poly
    cross = edges[:,0]*vecs[:,1] - edges[:,1]*vecs[:,0]
    return bool((cross >= 0).all())

def _seg_dist(p1, p2, p3, p4):
    EPS = 1e-12
    d1 = p2-p1; d2 = p4-p3; rv = p1-p3
    a = d1@d1; e = d2@d2; b = d1@d2; c = d1@rv; f = d2@rv
    denom = a*e - b*b
    s = float(np.clip((b*f-c*e)/denom,0,1)) if denom>EPS else 0.0
    t = float(np.clip((b*s+f)/e,0,1)) if e>EPS else 0.0
    s = float(np.clip((b*t-c)/a,0,1)) if a>EPS else 0.0
    return float(np.linalg.norm(p1+s*d1-p3-t*d2))

def reference(capsule, poly):
    p1, p2, r = capsule[:2], capsule[2:4], capsule[4]
    if _inside_convex(p1, poly) or _inside_convex(p2, poly):
        return True
    V = len(poly)
    for i in range(V):
        if _seg_dist(p1, p2, poly[i], poly[(i+1)%V]) <= r:
            return True
    return False

rng = np.random.default_rng(7)
def rand_square(rng):
    x, y = rng.uniform(0, 8), rng.uniform(0, 8)
    s = rng.uniform(1, 3)
    return np.array([[x,y],[x+s,y],[x+s,y+s],[x,y+s]])

t0 = time.time()
for _ in range(50):
    poly = rand_square(rng)
    pts = rng.uniform(0, 10, (2, 2))
    r = float(rng.uniform(0.2, 2.0))
    cap = np.array([pts[0,0], pts[0,1], pts[1,0], pts[1,1], r])
    expected = reference(cap, poly)
    got = capsule_poly_overlap(cap, poly)
    assert got == expected, f"Mismatch: expected {expected}, got {got}"
elapsed = time.time() - t0
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 50 random pairs vs reference ({elapsed:.3f}s)")

print("\nAll tests passed!")